# Treino de especialista — Google Colab

Escolha o agente na célula de configuração e execute tudo (`Ambiente de execução > Executar tudo`).

Antes de rodar: `Ambiente de execução > Alterar tipo de ambiente de execução > T4 GPU`.

**Nunca** envie o arquivo `.env`, chaves de API ou tokens para este notebook. O treino usa apenas dados históricos públicos.

In [ ]:
# ---------------------------------------------------------------- CONFIG
AGENT = "bull"        # bull | bear | ranger
TIMESTEPS = 300_000
SAVE_TO_DRIVE = True  # salva os artefatos no seu Google Drive ao final
# -------------------------------------------------------------------------
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "SEM GPU — ative T4 em Ambiente de execucao")

In [ ]:
# Clone raso: o repositorio guarda os parquets historicos e o historico completo
# passa de 480 MB. --depth 1 traz so a arvore atual.
#
# O %cd /content vem ANTES do rm: numa reexecucao o kernel ja esta dentro da
# pasta clonada, e apagar o proprio diretorio atual faz o git clone falhar.
%cd /content
!rm -rf /content/BinanceFuturesTrader
!git clone --depth 1 https://github.com/drtassio/BinanceFuturesTrader.git /content/BinanceFuturesTrader
%cd /content/BinanceFuturesTrader
!pip install -q -r cloud/requirements-cloud.txt
import os
assert os.path.exists('cloud/train_agent.py'), 'clone falhou: pare aqui'


In [ ]:
# Reconstroi o dataset causal a partir dos dados versionados.
# Barato (~1 min) e garante que o treino usa exatamente o mesmo codigo de
# features que o bot usa ao vivo.
import subprocess, sys
for script in ('scripts/build_causal_dataset.py', 'scripts/build_meta_features.py'):
    result = subprocess.run([sys.executable, script])
    # '!python' nao interrompe o notebook quando falha; aqui a falha para tudo.
    assert result.returncode == 0, f'{script} falhou'


In [ ]:
# Verificacao de causalidade. Se esta celula falhar, NAO treine:
# significa que alguma feature ainda enxerga o futuro.
import subprocess, sys
assert subprocess.run([sys.executable, 'scripts/verify_causality.py']).returncode == 0, 'vazamento detectado'


In [ ]:
import subprocess, sys
cmd = [sys.executable, "cloud/train_agent.py", "--agent", AGENT, "--timesteps", str(TIMESTEPS)]
print(" ".join(cmd))
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
print("exit code:", process.wait())
assert process.returncode == 0, 'treino falhou'


In [ ]:
import json, pathlib
report = json.loads(pathlib.Path(f"cloud/artifacts/{AGENT}_training_report.json").read_text())
verdict = report["verdict"]
print("APROVADO PARA OPERAR:", verdict["approved_for_live_trading"])
for name, passed in verdict["checks"].items():
    print(f"  [{'OK ' if passed else 'NAO'}] {name}")
print()
print(f"trades no holdout : {verdict['holdout_trades']}")
print(f"retorno liquido   : {verdict['holdout_net_return']*100:+.2f}%")
print(f"buy & hold        : {verdict['buy_and_hold_return']*100:+.2f}%")
print(f"drawdown maximo   : {verdict['holdout_max_drawdown']*100:.1f}%")
print()
det, sto = report["holdout_metrics"], report["holdout_metrics_stochastic"]
print(f"voto medio deterministico: {det['vote_mean']:+.3f} (desvio {det['vote_std']:.3f})")
print(f"voto medio estocastico   : {sto['vote_mean']:+.3f} (desvio {sto['vote_std']:.3f})")
print("Desvio deterministico ~0 = politica colapsada em uma unica decisao.")

In [ ]:
import shutil, pathlib, datetime
stamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%S")
archive = f"/content/{AGENT}_artifacts_{stamp}"
staging = pathlib.Path("/content/staging"); staging.mkdir(exist_ok=True)
for pattern in (f"models_ai/{AGENT}_specialist_sac.zip", f"models_ai/{AGENT}_specialist_scaler.joblib",
                f"cloud/artifacts/{AGENT}_training_report.json"):
    source = pathlib.Path(pattern)
    if source.exists():
        shutil.copy(source, staging / source.name)
for contract in pathlib.Path("cloud/artifacts").glob(f"{AGENT}_*/feature_contract.json"):
    shutil.copy(contract, staging / "feature_contract.json")
shutil.make_archive(archive, "gztar", staging)
print("pacote:", archive + ".tar.gz")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = pathlib.Path("/content/drive/MyDrive/BotShield"); destination.mkdir(parents=True, exist_ok=True)
    shutil.copy(archive + ".tar.gz", destination)
    print("salvo em", destination)

from google.colab import files
files.download(archive + ".tar.gz")